# Article 04 — Conducting a Manual FAIR Assessment
## Step-by-Step Walkthrough on a Real Pharmaceutical Dataset

**Series:** FAIR Data Maturity Framework  
**Author:** Ali Shahmohammadi, Ph.D. — Takeda Pharmaceutical  
**Prerequisites:** Articles 01, 02, 03

---

## Learning Objectives

1. Conduct a complete manual FAIR assessment using the RDA 41 indicators
2. Collect and document the evidence for each compliance decision
3. Generate a FAIR scorecard and gap analysis report
4. Prioritise remediation actions by impact and effort
5. Export the assessment as a reusable record (CSV, JSON)

---

## The Dataset We Are Assessing

**Dataset:** Cell Viability Assay Dataset — CAR-T Manufacturing Process Development  
**Internal ID:** STD-2024-CART-001  
**Description:** Trypan blue exclusion viability data for CAR-T cell manufacturing process optimisation.  
Three batches, 15 timepoints each. Raw Vi-CELL XR output + processed CSV.
**Current storage:** LabKey LIMS, internal only.  
**Current status:** No global PID, no external registration, internal use only.

In [ ]:
# ── Step 1: Initialise the assessor ───────────────────────────────────────
from fair_toolkit import ManualFAIRAssessor, ComplianceScore, EvidenceLevel

assessor = ManualFAIRAssessor(
    dataset_id="STD-2024-CART-001",
    dataset_title="CAR-T Cell Viability Assay — Process Development Batches 1-3",
    assessed_by="Ali Shahmohammadi, Ph.D.",
    notes="Internal dataset, LabKey LIMS. Assessing baseline prior to FAIRification roadmap.",
)

print(f"Assessor initialised for: {assessor.dataset_title}")
print(f"Total indicators to assess: {assessor.progress()['total']}")
print(f"Starting status: {assessor.progress()['assessed']} assessed, {assessor.progress()['remaining']} remaining")

In [ ]:
# ── Step 2: Assess Findable indicators (F1–F4) ─────────────────────────────
# F1: Persistent, globally unique identifiers

assessor.score(
    "RDA-F1-01D", ComplianceScore.NOT_COMPLIANT,
    evidence="Dataset identified by internal LabKey ID 'STD-2024-CART-001' only. "
             "This is not a globally unique PID — not registered with any PID authority.",
    notes="Action: Register dataset DOI via institutional DataCite account.",
)
assessor.score(
    "RDA-F1-01M", ComplianceScore.NOT_COMPLIANT,
    evidence="Metadata stored only in LabKey LIMS. No external metadata record with a PID.",
)
assessor.score(
    "RDA-F1-02D", ComplianceScore.NOT_COMPLIANT,
    evidence="Internal LIMS ID is not globally unique — another organisation could use the same ID.",
)
assessor.score(
    "RDA-F1-02M", ComplianceScore.NOT_COMPLIANT,
)

# F2: Rich metadata
assessor.score(
    "RDA-F2-01M", ComplianceScore.PARTIALLY_COMPLIANT,
    evidence="LabKey captures: title, creator, date, project code, data type. "
             "Missing: assay method ontology term (OBI), cell line ID (Cellosaurus), "
             "instrument model, batch effects, QC indicators (Z-prime, CV).",
    notes="Action: Map key fields to OBI/Cellosaurus. Add QC indicators. Add assay conditions.",
)

# F3: Metadata explicitly includes data identifier
assessor.score(
    "RDA-F3-01M", ComplianceScore.NOT_COMPLIANT,
    evidence="No explicit machine-readable link from metadata to data identifier. "
             "LabKey UI shows the data but metadata record has no typed relatedIdentifier field.",
)

# F4: Indexed in searchable resource
assessor.score(
    "RDA-F4-01M", ComplianceScore.NOT_COMPLIANT,
    evidence="Dataset not registered in any external catalogue "
             "(DataCite, Google Dataset Search, Zenodo, internal data catalogue).",
    notes="Action: Register in enterprise data catalogue (Azure Purview) as minimum.",
)

print("✓ Findable indicators assessed")
print(f"  F score so far: {assessor.build_result().f_score.percentage}%")

In [ ]:
# ── Step 3: Assess Accessible indicators (A) ──────────────────────────────

assessor.score(
    "RDA-A1-01M", ComplianceScore.PARTIALLY_COMPLIANT,
    evidence="Metadata in LabKey includes a contact email for the data owner. "
             "However, this is not machine-readable access information.",
)
assessor.score(
    "RDA-A1-02M", ComplianceScore.NOT_COMPLIANT,
    evidence="LabKey has a REST API but it requires an internal authentication token "
             "and is not documented publicly.",
)
assessor.score(
    "RDA-A1-03D", ComplianceScore.NOT_COMPLIANT,
    evidence="Data only accessible by logging into LabKey with corporate credentials. "
             "No programmatic retrieval by identifier.",
)
assessor.score(
    "RDA-A1-04M", ComplianceScore.NOT_COMPLIANT,
    evidence="LabKey metadata not accessible via any standardised external protocol.",
)
assessor.score(
    "RDA-A1.1-01M", ComplianceScore.NOT_APPLICABLE,
    reason="Dataset not yet accessible via any external protocol — indicator N/A.",
)
assessor.score(
    "RDA-A1.1-01D", ComplianceScore.NOT_APPLICABLE,
    reason="Same as above.",
)
assessor.score(
    "RDA-A1.2-01M", ComplianceScore.COMPLIANT,
    evidence="Internal access controlled via Azure AD SSO — standards-based OAuth 2.0.",
)
assessor.score(
    "RDA-A1.2-01D", ComplianceScore.COMPLIANT,
    evidence="Data access requires Azure AD authentication — corporate SSO is standards-based.",
)
assessor.score(
    "RDA-A2-01M", ComplianceScore.NOT_COMPLIANT,
    evidence="If LabKey were decommissioned, the metadata would be lost. "
             "No external metadata deposit exists.",
    notes="Action: Deposit metadata to DataCite as a minimum metadata preservation step.",
)

print("✓ Accessible indicators assessed")
print(f"  A score so far: {assessor.build_result().a_score.percentage}%")

In [ ]:
# ── Step 4: Assess Interoperable & Reusable indicators ────────────────────

# Interoperable
for iid, score, evidence in [
    ("RDA-I1-01M", ComplianceScore.NOT_COMPLIANT,
     "Metadata in LabKey proprietary XML with no public schema."),
    ("RDA-I1-01D", ComplianceScore.PARTIALLY_COMPLIANT,
     "Data exported as CSV — open format. No formal schema or data dictionary published."),
    ("RDA-I1-02M", ComplianceScore.NOT_COMPLIANT,
     "Metadata not machine-understandable without LabKey-specific knowledge."),
    ("RDA-I1-02D", ComplianceScore.PARTIALLY_COMPLIANT,
     "CSV is machine-parseable but no schema defines column semantics."),
    ("RDA-I2-01M", ComplianceScore.NOT_COMPLIANT,
     "Metadata uses internal codebook terms (e.g., 'VIAB' for viability) — not OBO ontology."),
    ("RDA-I2-01D", ComplianceScore.NOT_COMPLIANT,
     "Data column headers are internal abbreviations, not ontology-linked."),
    ("RDA-I3-01M", ComplianceScore.NOT_COMPLIANT, "No typed cross-references in metadata."),
    ("RDA-I3-01D", ComplianceScore.NOT_COMPLIANT, "No cross-references in data files."),
    ("RDA-I3-02M", ComplianceScore.NOT_COMPLIANT, None),
    ("RDA-I3-02D", ComplianceScore.NOT_COMPLIANT, None),
    ("RDA-I3-03M", ComplianceScore.NOT_COMPLIANT,
     "No reference to assay protocol DOI or instrument RRID."),
    ("RDA-I3-04M", ComplianceScore.NOT_COMPLIANT, "No metadata schema/profile declared."),
]:
    assessor.score(iid, score, evidence=evidence)

# Reusable
for iid, score, evidence, notes in [
    ("RDA-R1-01M", ComplianceScore.PARTIALLY_COMPLIANT,
     "Basic metadata present (title, date, creator). Missing: assay conditions, known limitations, QC stats.",
     "Action: Complete assay metadata to ISA-Tab minimum information standard."),
    ("RDA-R1.1-01M", ComplianceScore.NOT_COMPLIANT,
     "No licence specified for this dataset.", "Action: Apply CC BY 4.0 if pre-competitive, or 'Internal Use Only' if proprietary."),
    ("RDA-R1.1-02M", ComplianceScore.NOT_COMPLIANT, None, None),
    ("RDA-R1.1-03M", ComplianceScore.NOT_COMPLIANT, None, None),
    ("RDA-R1.2-01M", ComplianceScore.NOT_COMPLIANT,
     "No PROV-O or cross-domain provenance standard used.",
     "Action: Add W3C PROV metadata fields when creating external metadata record."),
    ("RDA-R1.2-02M", ComplianceScore.COMPLIANT,
     "Benchling ELN and LabKey both maintain 21 CFR Part 11 audit trails. ALCOA+ compliant.", None),
    ("RDA-R1.3-01M", ComplianceScore.NOT_COMPLIANT,
     "No community metadata standard (ISA-Tab, MIABI) applied.",
     "Action: Map to ISA-Tab Investigation/Study/Assay structure."),
    ("RDA-R1.3-01D", ComplianceScore.PARTIALLY_COMPLIANT,
     "CSV format is acceptable but Vi-CELL XR native format is proprietary binary.",
     "Action: Archive open-format CSV alongside raw instrument data."),
    ("RDA-R1.3-02M", ComplianceScore.NOT_COMPLIANT, None, None),
    ("RDA-R1.3-02D", ComplianceScore.NOT_COMPLIANT,
     "CSV not validated against any schema.", "Action: Publish data dictionary as schema.org/csvw."),
]:
    assessor.score(iid, score, evidence=evidence, notes=notes)

print("✓ All 41 indicators assessed")
print(f"  Progress: {assessor.progress()}")

In [ ]:
# ── Step 5: Generate the full FAIR scorecard ──────────────────────────────
assessor.print_scorecard()

In [ ]:
# ── Step 6: Export the assessment ─────────────────────────────────────────
import json, pandas as pd

result = assessor.build_result()

# Summary CSV
summary = result.summary_dict()
pd.DataFrame([summary]).to_csv("fair_assessment_summary.csv", index=False)
print("Summary saved to fair_assessment_summary.csv")
print()
print("Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

# Full indicator-level CSV
rows = []
for dim in [result.f_score, result.a_score, result.i_score, result.r_score]:
    for s in dim.indicator_scores:
        rows.append({
            "indicator_id": s.indicator_id,
            "principle": s.principle.value,
            "priority": s.priority.value,
            "compliance": s.compliance.value,
            "numeric_score": s.numeric_score,
            "evidence": s.evidence or "",
            "notes": s.notes or "",
        })
detail_df = pd.DataFrame(rows)
detail_df.to_csv("fair_assessment_detail.csv", index=False)
print("\nDetail CSV saved to fair_assessment_detail.csv")
print(f"Rows: {len(detail_df)}")

In [ ]:
# ── Step 7: Prioritised remediation roadmap ───────────────────────────────
result = assessor.build_result()

print("=" * 65)
print("FAIR REMEDIATION ROADMAP — STD-2024-CART-001")
print("=" * 65)

print("\n📍 Phase 1 — Quick Wins (1–2 weeks) — Address Essential gaps")
essential_gaps = result.get_gaps(EvidenceLevel.ESSENTIAL)
for gap in essential_gaps:
    print(f"  [{gap.principle.value}] {gap.indicator_id}: {gap.indicator_name[:55]}")
    if gap.notes:
        print(f"       → {gap.notes}")

print("\n📍 Phase 2 — Standards Adoption (1–3 months)")
print("  [F] Register dataset DOI via institutional DataCite account")
print("  [I] Map assay_type, cell_line, organism to OBI/Cellosaurus/NCBITaxon")
print("  [I] Publish data dictionary as schema.org/csvw")
print("  [R] Apply CC BY 4.0 licence for future external sharing")

print("\n📍 Phase 3 — Automation (3–6 months)")
print("  [F] Integrate enterprise data catalogue (Azure Purview) with LabKey")
print("  [A] Document and publish LabKey REST API externally")
print("  [I] Map LabKey metadata export to DCAT-AP schema")
print("  [R] Implement ISA-Tab metadata template in ELN")

print(f"\nEstimated FAIR score after Phase 1: ~35–45%")
print(f"Estimated FAIR score after Phase 2: ~55–65%")
print(f"Estimated FAIR score after Phase 3: ~70–80%")
print(f"Current FAIR score:                 {result.overall_score}%")